# 08 — Optimisation & Régularisation — BiLSTM

**Prérequis** : `06_evaluation_comparison.ipynb` exécuté → BiLSTM retenu comme modèle de déploiement (97.87% test accuracy).

**Données** : `../data/client_data.pkl`, `../data/vocab.pkl`, `../data/label_maps.json`

**Sortie** : `../models/bilstm_optimised.pt`

**Structure (pattern TP8)** :
```
Partie 0  : Config & chargement
Partie 1  : Baseline — BiLSTM sans optimisation (référence)
Partie 1b : Comparaison d'optimiseurs (SGD vs Adam vs AdamW)
Partie 2  : Schedulers LR (ReduceLROnPlateau vs CosineAnnealingLR)
Partie 3  : Early Stopping & sauvegarde du meilleur état
Partie 4  : Régularisation (Dropout, Weight Decay, num_layers)
Partie 5  : Entraînement complet assemblé
Partie 6  : Recherche d'hyperparamètres (Grid vs Random Search)
Partie 7  : Évaluation finale
```

## Partie 0 — Config & chargement

In [ ]:
import os, pickle, json, copy, time, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR  = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

# Hyperparamètres baseline (identiques au notebook 03)
EMBED_DIM    = 128
HIDDEN_DIM   = 256
DROPOUT      = 0.3
BATCH_SIZE   = 64
LR           = 5e-4
N_EPOCHS     = 15
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

In [ ]:
# Chargement des données
with open(DATA_DIR + 'client_data.pkl', 'rb') as f:
    data = pickle.load(f)
with open(DATA_DIR + 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
N_CLASSES  = len(CLIENT_LABELS)
label_names = [CLIENT_LABELS[i] for i in range(N_CLASSES)]
PAD_IDX = vocab['<PAD>']

print(f'Vocab : {len(vocab):,} tokens')
print(f'Train : {len(data["train"]):,} | Val : {len(data["val"]):,} | Test : {len(data["test"]):,}')
print(f'Classes ({N_CLASSES}) : {label_names}')

In [ ]:
# Dataset + DataLoader (pattern TP6)
class IntentDataset(Dataset):
    def __init__(self, data):
        self.texts  = [d[0] for d in data]
        self.labels = [d[1] for d in data]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded  = pad_sequence(
        [torch.tensor(t, dtype=torch.long) for t in texts],
        batch_first=True, padding_value=PAD_IDX
    )
    return texts_padded, torch.tensor(labels, dtype=torch.long)

loaders = {
    split: DataLoader(
        IntentDataset(data[split]),
        batch_size=BATCH_SIZE, shuffle=(split == 'train'), collate_fn=collate_fn
    )
    for split in ('train', 'val', 'test')
}
x, y = next(iter(loaders['train']))
print(f'Batch — textes: {x.shape}, labels: {y.shape}')

In [ ]:
# Architecture BiLSTM (identique au notebook 03)
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim,
                 num_layers=1, dropout=0.3):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim,
                                 num_layers=num_layers, batch_first=True,
                                 bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded       = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        last_hidden    = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(last_hidden))


def build_bilstm(embed_dim=128, hidden_dim=256, dropout=0.3, num_layers=1):
    return BiLSTM(len(vocab), embed_dim, hidden_dim, N_CLASSES, num_layers, dropout).to(device)

# Vérification
m_test = build_bilstm()
print(f'Paramètres : {sum(p.numel() for p in m_test.parameters()):,}')

In [ ]:
# Fonctions communes (pattern TP8)
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct    += (outputs.argmax(dim=1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs    = model(texts)
            loss       = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct    += (outputs.argmax(dim=1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def plot_history(histories, labels, title='Courbes entraînement'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    colors = ['steelblue', 'coral', 'seagreen', 'orchid']
    for i, (h, lbl) in enumerate(zip(histories, labels)):
        ep = range(1, len(h['val_loss']) + 1)
        c  = colors[i % len(colors)]
        axes[0].plot(ep, h['train_loss'], '--', color=c, alpha=0.6, label=f'{lbl} train')
        axes[0].plot(ep, h['val_loss'],   '-',  color=c, label=f'{lbl} val')
        axes[1].plot(ep, h['train_acc'],  '--', color=c, alpha=0.6)
        axes[1].plot(ep, h['val_acc'],    '-',  color=c, label=lbl)
        if h.get('stopped_epoch'):
            for ax in axes:
                ax.axvline(h['stopped_epoch'], color=c, linestyle=':', alpha=0.7)
    for ax, ylabel in zip(axes, ['Loss', 'Accuracy']):
        ax.set_xlabel('Époque'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    axes[0].set_title('Loss'); axes[1].set_title('Accuracy')
    plt.suptitle(title); plt.tight_layout(); plt.show()

## Partie 1 — Baseline BiLSTM (pattern TP8)

Ré-entraînement **sans aucune optimisation** — même config que notebook 03.
Sert de référence pour mesurer l'apport de chaque technique.

In [ ]:
def run_training_basic(model, loaders, criterion, optimizer, num_epochs):
    """Boucle minimaliste : pas de scheduler, pas d'early stopping (pattern TP8 Partie 1)."""
    model = model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(num_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, loaders['train'], criterion, optimizer)
        va_loss, va_acc = evaluate(model, loaders['val'], criterion)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:2d}/{num_epochs} | '
              f'train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
              f'val loss {va_loss:.4f} acc {va_acc:.4f} | LR {lr:.2e} | {time.time()-t0:.0f}s')
    history['best_val_acc'] = max(history['val_acc'])
    return model, history


criterion       = nn.CrossEntropyLoss()
model_baseline  = build_bilstm()
opt_baseline    = optim.Adam(model_baseline.parameters(), lr=LR)

print('Baseline BiLSTM (LR fixe, pas de scheduler)')
model_baseline, history_baseline = run_training_basic(
    model_baseline, loaders, criterion, opt_baseline, N_EPOCHS
)
print(f'Best val_acc baseline : {history_baseline["best_val_acc"]:.4f}')

## Partie 1b — Comparaison d'optimiseurs (SGD vs Adam vs AdamW)

Même architecture et mêmes hyperparamètres que le baseline (`LR`, `dropout`), seul l'optimiseur change.
Budget limité à **5 époques** par optimiseur pour comparer leur vitesse de convergence et leur val_acc.

| Optimiseur | Principe |
|---|---|
| `SGD` (momentum=0.9) | Descente de gradient classique + momentum pour accélérer la convergence |
| `Adam` | Moments adaptatifs (moyenne + variance des gradients), convergence rapide |
| `AdamW` | Comme Adam mais weight decay découplé de la mise à jour du gradient (meilleure régularisation) |

In [ ]:
N_EPOCHS_OPT = 5

optimizers_cfg = {
    'SGD':   lambda m: optim.SGD(m.parameters(), lr=LR, momentum=0.9),
    'Adam':  lambda m: optim.Adam(m.parameters(), lr=LR),
    'AdamW': lambda m: optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4),
}

histories_opt = {}
for name, make_opt in optimizers_cfg.items():
    print(f'\n=== Optimiseur : {name} ===')
    model_opt = build_bilstm()
    opt       = make_opt(model_opt)
    _, h      = run_training_basic(model_opt, loaders, criterion, opt, N_EPOCHS_OPT)
    histories_opt[name] = h
    print(f'{name} — best val_acc : {h["best_val_acc"]:.4f}')

print('\n=== Comparaison des optimiseurs ===')
for name, h in histories_opt.items():
    print(f'{name:<6} → best val_acc = {h["best_val_acc"]:.4f}')

plot_history(
    list(histories_opt.values()),
    list(histories_opt.keys()),
    'Comparaison des optimiseurs — SGD vs Adam vs AdamW (5 époques)'
)

## Partie 2 — Schedulers de Learning Rate (pattern TP8)

| Scheduler | Principe | Profil LR |
|---|---|---|
| `ReduceLROnPlateau` | Divise LR si val_loss stagne `patience` époques | Escalier |
| `CosineAnnealingLR` | Décroissance cosinus sur `T_max` époques | Continu |

> Pas de LinearWarmup ici — c'est spécifique aux modèles pré-entraînés (BERT).
> Le BiLSTM est entraîné from scratch → un LR fixe ou schedulé classique suffit.

In [ ]:
def run_training_with_scheduler(model, loaders, criterion, optimizer, num_epochs, scheduler=None):
    model = model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(num_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, loaders['train'], criterion, optimizer)
        va_loss, va_acc = evaluate(model, loaders['val'], criterion)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_loss)
            else:
                scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:2d}/{num_epochs} | '
              f'train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
              f'val loss {va_loss:.4f} acc {va_acc:.4f} | LR {lr:.2e} | {time.time()-t0:.0f}s')
    history['best_val_acc'] = max(history['val_acc'])
    return model, history

In [ ]:
# Stratégie 1 : ReduceLROnPlateau
model_plateau = build_bilstm()
opt_plateau   = optim.Adam(model_plateau.parameters(), lr=LR)
sch_plateau   = optim.lr_scheduler.ReduceLROnPlateau(
    opt_plateau, mode='min', patience=2, factor=0.5)

print('Stratégie 1 : ReduceLROnPlateau')
model_plateau, history_plateau = run_training_with_scheduler(
    model_plateau, loaders, criterion, opt_plateau, N_EPOCHS, sch_plateau)

In [ ]:
# Stratégie 2 : CosineAnnealingLR
model_cosine = build_bilstm()
opt_cosine   = optim.Adam(model_cosine.parameters(), lr=LR)
sch_cosine   = optim.lr_scheduler.CosineAnnealingLR(opt_cosine, T_max=N_EPOCHS)

print('Stratégie 2 : CosineAnnealingLR')
model_cosine, history_cosine = run_training_with_scheduler(
    model_cosine, loaders, criterion, opt_cosine, N_EPOCHS, sch_cosine)

# Visualisation profils LR théoriques
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
lr_plateau_sim = [LR]*5 + [LR/2]*5 + [LR/4]*5
axes[0].step(range(1, N_EPOCHS+1), lr_plateau_sim, color='steelblue', where='post')
axes[0].set_title('ReduceLROnPlateau (escalier)'); axes[0].set_xlabel('Époque')
epochs_cos = np.arange(1, N_EPOCHS+1)
lr_cos_sim = LR * (1 + np.cos(np.pi * (epochs_cos-1) / N_EPOCHS)) / 2
axes[1].plot(epochs_cos, lr_cos_sim, color='coral')
axes[1].set_title('CosineAnnealingLR (continu)'); axes[1].set_xlabel('Époque')
plt.suptitle('Profils de Learning Rate'); plt.tight_layout(); plt.show()

plot_history(
    [history_baseline, history_plateau, history_cosine],
    ['Baseline', 'ReduceLROnPlateau', 'CosineAnnealing'],
    'Comparaison des schedulers — BiLSTM'
)

## Partie 3 — Early Stopping & sauvegarde du meilleur modèle (pattern TP8)

L'early stopping arrête l'entraînement quand la val_acc ne s'améliore plus pendant `patience` époques
et **restaure les poids du meilleur état** (pas forcément le dernier).

> **Différence scheduler vs early stopping** :
> - Le scheduler *adapte* le LR pour continuer malgré un plateau.
> - L'early stopping *arrête* pour éviter l'overfitting.

In [ ]:
def run_training_full(model, loaders, criterion, optimizer, scheduler=None,
                      num_epochs=15, patience=None):
    """Boucle complète : scheduler + early stopping + sauvegarde meilleur état (pattern TP8)."""
    model = model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc  = 0.0
    best_weights  = copy.deepcopy(model.state_dict())
    no_improve    = 0
    stopped_epoch = None

    for epoch in range(num_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, loaders['train'], criterion, optimizer)
        va_loss, va_acc = evaluate(model, loaders['val'], criterion)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_loss)
            else:
                scheduler.step()

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_weights = copy.deepcopy(model.state_dict())
            no_improve   = 0
            star = ' ✓'
        else:
            no_improve += 1
            star = f' (patience {no_improve}/{patience})' if patience else ''

        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:2d}/{num_epochs} | '
              f'train {tr_loss:.4f}/{tr_acc:.4f} | '
              f'val {va_loss:.4f}/{va_acc:.4f} | LR {lr:.2e} | {time.time()-t0:.0f}s{star}')

        if patience is not None and no_improve >= patience:
            stopped_epoch = epoch + 1
            print(f'\n[Early stopping] époque {stopped_epoch} (patience={patience})')
            break

    history['stopped_epoch'] = stopped_epoch
    history['best_val_acc']  = best_val_acc
    model.load_state_dict(best_weights)
    print(f'Meilleure val_acc : {best_val_acc:.4f}')
    return model, history

In [ ]:
# Early stopping + ReduceLROnPlateau
model_es = build_bilstm()
opt_es   = optim.Adam(model_es.parameters(), lr=LR)
sch_es   = optim.lr_scheduler.ReduceLROnPlateau(opt_es, mode='min', patience=2, factor=0.5)

print('Early Stopping (patience=4) + ReduceLROnPlateau')
model_es, history_es = run_training_full(
    model_es, loaders, criterion, opt_es, sch_es,
    num_epochs=20, patience=4
)
plot_history([history_es], ['Early Stopping'], 'Early stopping BiLSTM')
print(f'Arrêt à l\'époque : {history_es["stopped_epoch"]}')

## Partie 4 — Régularisation (pattern TP8)

| Technique | Application BiLSTM | Paramètre |
|---|---|---|
| **Dropout** | Dans l'architecture (entre couches LSTM + avant FC) | `dropout` (0.1 → 0.5) |
| **Weight Decay** | Dans l'optimiseur Adam | `weight_decay` (0 → 1e-3) |
| **num_layers** | Couches LSTM empilées | `num_layers` (1 → 2) |

> Le BiLSTM ne contient pas de BatchNorm — le dropout est la principale régularisation.

In [ ]:
# Sans régularisation supplémentaire (dropout=0, weight_decay=0)
model_noreg = build_bilstm(dropout=0.0)
opt_noreg   = optim.Adam(model_noreg.parameters(), lr=LR, weight_decay=0.0)
sch_noreg   = optim.lr_scheduler.ReduceLROnPlateau(opt_noreg, patience=2, factor=0.5)
print('Sans régularisation (dropout=0, weight_decay=0)')
model_noreg, history_noreg = run_training_full(
    model_noreg, loaders, criterion, opt_noreg, sch_noreg, num_epochs=N_EPOCHS)

In [ ]:
# Avec Dropout=0.5 + Weight Decay=1e-4
model_reg = build_bilstm(dropout=0.5)
opt_reg   = optim.Adam(model_reg.parameters(), lr=LR, weight_decay=1e-4)
sch_reg   = optim.lr_scheduler.ReduceLROnPlateau(opt_reg, patience=2, factor=0.5)
print('Avec Dropout=0.5 + Weight Decay=1e-4')
model_reg, history_reg = run_training_full(
    model_reg, loaders, criterion, opt_reg, sch_reg, num_epochs=N_EPOCHS)

plot_history(
    [history_noreg, history_reg],
    ['Sans régularisation', 'Dropout=0.5 + WeightDecay'],
    'Impact de la régularisation — BiLSTM'
)

In [ ]:
# Inspection des couches BiLSTM (équivalent TP8 inspection BatchNorm)
print('Architecture BiLSTM :')
model_tmp = build_bilstm(num_layers=2)
for name, module in model_tmp.named_modules():
    if isinstance(module, (nn.LSTM, nn.Dropout, nn.Linear, nn.Embedding)):
        print(f'  {name:20s} : {module.__class__.__name__}')

# Vérification dropout actif en train, désactivé en eval
model_tmp.train()
x_test = torch.ones(1, 10, dtype=torch.long).to(next(model_tmp.parameters()).device)
out_train_1 = model_tmp(x_test)
out_train_2 = model_tmp(x_test)
model_tmp.eval()
out_eval_1  = model_tmp(x_test)
out_eval_2  = model_tmp(x_test)
print(f'\nTrain : sorties identiques ? {torch.allclose(out_train_1, out_train_2)} (doit être False — dropout aléatoire)')
print(f'Eval  : sorties identiques ? {torch.allclose(out_eval_1,  out_eval_2)}  (doit être True  — dropout désactivé)')

## Partie 5 — Entraînement complet assemblé (pattern TP8)

LR différencié par groupe de couches :
- **Embedding** : LR plus faible — les embeddings sont déjà bien initialisés après le baseline
- **LSTM** : LR standard
- **FC** : LR plus grand — tête de classification

In [ ]:
model_ft = build_bilstm(dropout=0.3, num_layers=2)

# LR différencié par groupe (pattern TP8 Partie 5)
opt_ft = optim.Adam([
    {'params': model_ft.embedding.parameters(), 'lr': LR * 0.5},  # embedding : LR faible
    {'params': model_ft.lstm.parameters(),      'lr': LR},         # LSTM : LR standard
    {'params': model_ft.fc.parameters(),        'lr': LR * 2}      # FC : LR plus grand
], weight_decay=1e-4)

sch_ft = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=20)

print('Entraînement complet : LR différencié + CosineAnnealing + Early Stopping')
model_ft, history_ft = run_training_full(
    model_ft, loaders, criterion, opt_ft, sch_ft,
    num_epochs=20, patience=5
)

torch.save(model_ft.state_dict(), MODEL_DIR + 'bilstm_optimised.pt')
print(f'Modèle optimisé sauvegardé → {MODEL_DIR}bilstm_optimised.pt')

plot_history(
    [history_baseline, history_ft],
    ['Baseline', 'Fine-tuning complet'],
    'Baseline vs Entraînement complet — BiLSTM'
)

## Partie 6 — Recherche d'hyperparamètres 

**Grid Search vs Random Search** sur les 3 hyperparamètres clés du BiLSTM.

Budget limité : 3 époques par combinaison pour identifier la région prometteuse.

In [ ]:
# Grid Search
lr_values      = [1e-3, 5e-4, 1e-4]
dropout_values = [0.1, 0.3, 0.5]
grid = list(itertools.product(lr_values, dropout_values))
print(f'Grid Search : {len(grid)} combinaisons')

results_grid = []
for lr, dropout in grid:
    print(f'\n--- lr={lr:.0e}, dropout={dropout} ---')
    m   = build_bilstm(dropout=dropout)
    opt = optim.Adam(m.parameters(), lr=lr)
    _, h = run_training_full(m, loaders, criterion, opt, num_epochs=3)
    results_grid.append({'lr': lr, 'dropout': dropout, 'val_acc': h['best_val_acc']})

results_grid.sort(key=lambda x: x['val_acc'], reverse=True)
print('\n=== Grid Search ===')
for r in results_grid:
    print(f'lr={r["lr"]:.0e}  dropout={r["dropout"]}  → val_acc={r["val_acc"]:.4f}')

In [ ]:
# Random Search 
N_TRIALS = len(grid)
results_random = []

for trial in range(N_TRIALS):
    lr      = 10 ** np.random.uniform(-4, -3)   # log-uniforme [1e-4, 1e-3]
    dropout = np.random.uniform(0.1, 0.5)
    print(f'\n--- Trial {trial+1}/{N_TRIALS} - lr={lr:.2e}, dropout={dropout:.2f} ---')
    m   = build_bilstm(dropout=dropout)
    opt = optim.Adam(m.parameters(), lr=lr)
    _, h = run_training_full(m, loaders, criterion, opt, num_epochs=3)
    results_random.append({'lr': lr, 'dropout': dropout, 'val_acc': h['best_val_acc']})

results_random.sort(key=lambda x: x['val_acc'], reverse=True)
print('\n=== Comparaison ===')
print(f'Grid   - meilleure val_acc : {results_grid[0]["val_acc"]:.4f}')
print(f'Random - meilleure val_acc : {results_random[0]["val_acc"]:.4f}')

In [ ]:
# Heatmap Grid Search
grid_matrix = np.zeros((len(lr_values), len(dropout_values)))
for r in results_grid:
    i = lr_values.index(r['lr'])
    j = dropout_values.index(r['dropout'])
    grid_matrix[i, j] = r['val_acc']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im = axes[0].imshow(grid_matrix, cmap='YlGn', vmin=grid_matrix.min()*0.999, vmax=1.0)
axes[0].set_xticks(range(len(dropout_values)))
axes[0].set_yticks(range(len(lr_values)))
axes[0].set_xticklabels([str(d) for d in dropout_values])
axes[0].set_yticklabels([f'{lr:.0e}' for lr in lr_values])
axes[0].set_xlabel('Dropout'); axes[0].set_ylabel('LR')
axes[0].set_title('Grid Search — Val Accuracy')
for i in range(len(lr_values)):
    for j in range(len(dropout_values)):
        axes[0].text(j, i, f'{grid_matrix[i,j]:.4f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=axes[0])

sc = axes[1].scatter(
    [r['dropout'] for r in results_random],
    [r['lr']      for r in results_random],
    c=[r['val_acc'] for r in results_random],
    cmap='YlGn', s=120, edgecolors='gray', linewidths=0.5
)
axes[1].set_yscale('log')
axes[1].set_xlabel('Dropout'); axes[1].set_ylabel('LR (log)')
axes[1].set_title('Random Search — Val Accuracy')
plt.colorbar(sc, ax=axes[1])
plt.tight_layout(); plt.show()

## Partie 7 — Évaluation finale

In [ ]:
# Évaluation sur le test set
model_ft.eval()
preds, targets = [], []
with torch.no_grad():
    for texts, labels in loaders['test']:
        texts = texts.to(device)
        preds.extend(model_ft(texts).argmax(dim=1).cpu().tolist())
        targets.extend(labels.tolist())

test_acc = accuracy_score(targets, preds)

# Chargement du résultat baseline pour comparaison
with open(DATA_DIR + 'results_bilstm.pkl', 'rb') as f:
    res_bilstm = pickle.load(f)

print(f'Test Accuracy — BiLSTM notebook 03 : {res_bilstm["test_accuracy"]:.4f}')
print(f'Test Accuracy — BiLSTM optimisé    : {test_acc:.4f}')
print(f'Gain : {(test_acc - res_bilstm["test_accuracy"])*100:+.2f} pts')

print('\n' + classification_report(targets, preds, target_names=label_names))

cm = confusion_matrix(targets, preds)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'Matrice de confusion — BiLSTM optimisé (acc={test_acc:.4f})')
plt.ylabel('Vrai'); plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'bilstm_optimised_confusion.png', dpi=100)
plt.show()

In [ ]:
# Tableau récapitulatif (pattern TP8)
print('='*55)
print(f'{"Configuration":<30} {"Best Val Acc":>12}')
print('='*55)
for name, h in histories_opt.items():
    print(f'{f"Optimiseur {name} (5 ep.)":<30} {h["best_val_acc"]:>12.4f}')
print(f'{"Baseline (LR fixe)":<30} {history_baseline["best_val_acc"]:>12.4f}')
print(f'{"ReduceLROnPlateau":<30} {history_plateau["best_val_acc"]:>12.4f}')
print(f'{"CosineAnnealingLR":<30} {history_cosine["best_val_acc"]:>12.4f}')
print(f'{"Early Stopping":<30} {history_es["best_val_acc"]:>12.4f}')
print(f'{"Sans régularisation":<30} {history_noreg["best_val_acc"]:>12.4f}')
print(f'{"Dropout + Weight Decay":<30} {history_reg["best_val_acc"]:>12.4f}')
print(f'{"Entraînement complet":<30} {history_ft["best_val_acc"]:>12.4f}')
print(f'{"TEST SET":<30} {test_acc:>12.4f}')
print('='*55)